# Orthomosaic — Part 4: PMTiles — Interactive Map Serving

**Convert the COG to PMTiles for interactive map serving.**

PMTiles is a single-file tile archive (EPSG:3857 raster tiles).  The conversion
pipeline:
1. Reproject COG → EPSG:3857 via `rasterio.warp`.
2. Tile the reprojected COG via `pymbtiles` (MBTiles) + `go-pmtiles convert`.
3. Copy to a UC Volume path for HTTP serving.
4. Preview with `render_pmtiles_preview`.

> **Prerequisite.** `03_cog` must have written the COG to `cog_dir`.

> **Runtime.** Runs on **Serverless environment 5**.

---

**Last Update:** September 21, 2026

## Setup

In [ ]:
%run ./config_nb

## Step 1: Reproject COG → Web Mercator (EPSG:3857)

In [ ]:
import time as _t_reproj
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling as _R
from pathlib import Path as _P

_t0 = _t_reproj.perf_counter()
try:
    _groups = discover_groups()
    if not _groups:
        raise RuntimeError(f"no group_* COGs under {output_dir} \u2014 run 01-03 first")
    for grp in _groups:
        _gp = group_paths(grp)
        _P(_gp["cog_3857"]).parent.mkdir(parents=True, exist_ok=True)
        with rasterio.open(_gp["cog"]) as src:
            _t3857, _w3857, _h3857 = calculate_default_transform(
                src.crs, "EPSG:3857", src.width, src.height, *src.bounds)
            profile = src.profile.copy()
            profile.update(crs="EPSG:3857", transform=_t3857, width=_w3857, height=_h3857, compress="lzw")
            with rasterio.open(_gp["cog_3857"], "w", **profile) as dst:
                for i in range(1, src.count + 1):
                    reproject(source=rasterio.band(src, i), destination=rasterio.band(dst, i),
                              src_transform=src.transform, src_crs=src.crs,
                              dst_transform=_t3857, dst_crs="EPSG:3857", resampling=_R.average)
        print(f"  group {grp!r}: reprojected \u2192 {_gp['cog_3857']}")
    print(f"Reprojected {len(_groups)} group(s) in {_t_reproj.perf_counter()-_t0:.1f}s")
except Exception as e:
    print(f"[ERROR] Reproject step failed after {_t_reproj.perf_counter()-_t0:.1f}s: {e}")
    raise

## Step 2: Convert to PMTiles

In [ ]:
import io as _io, math as _math
import time as _t_pm
import numpy as _np
import rasterio, shutil
import mercantile, pymbtiles
from PIL import Image as _PILImage
from pathlib import Path as _P
from rasterio.enums import Resampling as _R
from rasterio.warp import transform_bounds as _transform_bounds

_t0 = _t_pm.perf_counter()
try:
    _groups = discover_groups()
    if not _groups:
        raise RuntimeError(f"no group_* COGs under {output_dir} \u2014 run 01-03 first")
    _MIN_Z, _MAX_Z = 12, 21
    for grp in _groups:
        _gp = group_paths(grp)
        _cog3857 = _gp["cog_3857"]; _pm_local = _gp["pmtiles_local"]; _pm_vol = _gp["pmtiles"]
        _MB_PATH = _pm_local.replace(".pmtiles", ".mbtiles")
        _P(_pm_local).parent.mkdir(parents=True, exist_ok=True)
        with rasterio.open(_cog3857) as _src:
            _b = _src.bounds  # EPSG:3857 metres
            _wd, _sd, _ed, _nd = _transform_bounds("EPSG:3857", "EPSG:4326",
                                                   _b.left, _b.bottom, _b.right, _b.top)
            _res = (_b.right - _b.left) / _src.width
            _lat = _math.radians((_sd + _nd) / 2.0)
            _zoom = max(_MIN_Z, min(_MAX_Z, int(round(_math.log2(156543.03392 * _math.cos(_lat) / _res)))))
        _n_tiles = 0
        with pymbtiles.MBtiles(_MB_PATH, "w") as _db:
            _db.meta = {"name": f"orthomosaic_{grp}", "format": "jpeg",
                        "bounds": f"{_wd},{_sd},{_ed},{_nd}", "type": "overlay"}
            with rasterio.open(_cog3857) as _src:
                for _tile in mercantile.tiles(_wd, _sd, _ed, _nd, zooms=_zoom):
                    _xb = mercantile.xy_bounds(_tile)
                    _win = _src.window(_xb.left, _xb.bottom, _xb.right, _xb.top)
                    try:
                        _data = _src.read(window=_win, out_shape=(3, 256, 256),
                                          resampling=_R.average, boundless=True, fill_value=0)
                    except Exception:
                        continue
                    if not _data.any():
                        continue
                    _img = _PILImage.fromarray(_data.transpose(1, 2, 0), "RGB")
                    _buf = _io.BytesIO(); _img.save(_buf, format="JPEG", quality=82)
                    _db.write_tile(_tile.z, _tile.x, _tile.y, _buf.getvalue()); _n_tiles += 1
        if _n_tiles == 0:
            raise RuntimeError(f"group {grp!r}: 0 tiles written \u2014 check COG bounds/zoom")
        _conv = subprocess.run([str(PMTILES_BIN), "convert", _MB_PATH, _pm_local],
                               capture_output=True, text=True)
        if _conv.returncode != 0:
            raise RuntimeError(f"group {grp!r}: go-pmtiles convert failed (rc={_conv.returncode}): "
                               f"{(_conv.stderr or _conv.stdout).strip()[-400:]}")
        _P(_pm_vol).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(_pm_local, _pm_vol)
        print(f"  group {grp!r}: {_n_tiles} tiles \u2192 PMTiles \u2192 {_pm_vol}")
    print(f"PMTiles for {len(_groups)} group(s) in {_t_pm.perf_counter()-_t0:.1f}s")
except Exception as e:
    print(f"[ERROR] PMTiles step failed after {_t_pm.perf_counter()-_t0:.1f}s: {e}")
    raise

## Step 3: In-notebook Preview

In [ ]:
vz.plot_pmtiles(group_paths(discover_groups()[0])["pmtiles"])

## Steps performed

1. **Reprojection** — COG reprojected to EPSG:3857 via `rasterio.warp`.
2. **Tiling** — rasterised at the auto-detected zoom level into JPEG tiles; assembled via `pymbtiles`.
3. **PMTiles conversion** — `go-pmtiles convert` produced a single-file archive.
4. **Volume copy** — copied to UC Volume for HTTP serving.
5. **Preview** — `render_pmtiles_preview` stitched tiles into a Folium map.

**Series complete.** The orthomosaic GeoTIFF, COG, and PMTiles are available in
`output_dir` and the configured UC Volume.